In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
def se_block_1d(x, ratio=2):
    ch = x.shape[-1]
    se = layers.GlobalAveragePooling1D()(x)
    se = layers.Reshape((1, ch))(se)
    se = layers.Conv1D(max(ch // ratio, 4), 1, activation="relu", use_bias=False)(se)
    se = layers.Conv1D(ch, 1, activation="sigmoid", use_bias=False)(se)
    return layers.Multiply()([x, se])

def ds_block_plain_1d(x, filters):
    x = layers.DepthwiseConv1D(3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.ReLU(6.)(x)
    x = layers.Conv1D(filters, 1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.ReLU(6.)(x)
    return x

def ds_se_residual_block_1d(x, filters, stride=1):
    in_ch = x.shape[-1]
    residual = x

    y = layers.DepthwiseConv1D(3, strides=stride, padding="same", use_bias=False)(x)
    y = layers.BatchNormalization(momentum=0.9)(y)
    y = layers.ReLU(6.)(y)
    y = layers.Conv1D(filters, 1, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization(momentum=0.9)(y)
    y = se_block_1d(y)

    if stride == 2:
        residual = layers.AveragePooling1D(2, strides=2, padding="same")(residual)
    if in_ch != filters:
        residual = layers.Conv1D(filters, 1, padding="same", use_bias=False)(residual)
        residual = layers.BatchNormalization(momentum=0.9)(residual)

    y = layers.Add()([y, residual])
    y = layers.ReLU(6.)(y)
    return y

def build_lightweight_fall_detector(input_shape=(300, 9)):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv1D(16, 5, strides=2, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.ReLU(6.)(x)

    x = ds_block_plain_1d(x, 16)
    x = ds_se_residual_block_1d(x, 32, stride=2)
    x = ds_se_residual_block_1d(x, 64, stride=2)
    x = ds_se_residual_block_1d(x, 64, stride=1)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)
    return model

model = build_lightweight_fall_detector()
model.compile(
    optimizer='adam', 
    loss='binary_crossentropy', 
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 9)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 150, 16)   │        720 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 150, 16)   │         64 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 150, 16)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv1d    │ (None, 150, 16)   │         48 │ re_lu[0][0]       │
│ (DepthwiseConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 150, 16)   │         64 │ depthwise_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 150, 16)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 150, 16)   │        256 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 150, 16)   │         64 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 150, 16)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv1d_1  │ (None, 75, 16)    │         48 │ re_lu_2[0][0]     │
│ (DepthwiseConv1D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 75, 16)    │         64 │ depthwise_conv1d… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 75, 16)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 75, 32)    │        512 │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 75, 32)    │        128 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 32)     │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 1, 16)     │        512 │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ average_pooling1d   │ (None, 75, 16)    │          0 │ re_lu_2[0][0]     │
│ (AveragePooling1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 1, 32)     │        512 │ conv1d_3[0][0]  

 Total params: 21,521 (84.07 KB)

 Trainable params: 20,689 (80.82 KB)

 Non-trainable params: 832 (3.25 KB)

In [ ]:

# 1. Khối SE Block 1 Chiều

def se_block_1d(x, ratio=2):
    """SE attention cho feature map 1D (time, channels)."""
    ch = x.shape[-1]
    se = layers.GlobalAveragePooling1D()(x)
    se = layers.Reshape((1, ch))(se)
    se = layers.Conv1D(max(ch // ratio, 4), 1, activation="relu", use_bias=False)(se)
    se = layers.Conv1D(ch, 1, activation="sigmoid", use_bias=False)(se)
    return layers.Multiply()([x, se])


# 2. Khối Depthwise Separable Plain 1 Chiều

def ds_block_plain_1d(x, filters):
    """Depthwise separable block dạng plain 1D."""
    x = layers.DepthwiseConv1D(3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.ReLU(6.0)(x)

    x = layers.Conv1D(filters, 1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization(momentum=0.9)(x)
    x = layers.ReLU(6.0)(x)
    return x


# 3. Khối Depthwise Separable Residual + SE 1 Chiều

def ds_se_residual_block_1d(x, filters, stride=1):
    """Residual block 1D sử dụng depthwise-separable + SE attention."""
    in_ch = x.shape[-1]
    residual = x

    y = layers.DepthwiseConv1D(3, strides=stride, padding="same", use_bias=False)(x)
    y = layers.BatchNormalization(momentum=0.9)(y)
    y = layers.ReLU(6.0)(y)

    y = layers.Conv1D(filters, 1, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization(momentum=0.9)(y)
    y = se_block_1d(y)

    if stride == 2:
        residual = layers.AveragePooling1D(2, strides=2, padding="same")(residual)
    if in_ch != filters:
        residual = layers.Conv1D(filters, 1, padding="same", use_bias=False)(residual)
        residual = layers.BatchNormalization(momentum=0.9)(residual)

    y = layers.Add()([y, residual])
    y = layers.ReLU(6.0)(y)
    return y

# Xây dựng kiến trúc mô hình Deep Learning (MLP/1D CNN hybrid)
inputs = Input(shape=(X_train.shape[1],), name="input_sensor_features")
x = layers.Reshape((WINDOW_SIZE, len(feature_cols)))(inputs)

x = ds_block_plain_1d(x, filters=32)
x = ds_se_residual_block_1d(x, filters=64)
x = ds_se_residual_block_1d(x, filters=64, stride=2)

x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.35)(x)

x = layers.Dense(32, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)

outputs = Dense(1, activation="sigmoid", name="fall_prediction")(x)

model = Model(inputs=inputs, outputs=outputs, name="Fall_Detection_1D_CNN_SE")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()


NameError: name 'Input' is not defined